In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Configuración del entorno (evitar problemas de compatibilidades)
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['TF_GPU_THREAD_MODE'] = 'gpu_private'
os.environ['TF_GPU_THREAD_COUNT'] = '1'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, optimizers, callbacks, regularizers
from tensorflow.keras.applications import EfficientNetB0
from transformers import AutoTokenizer, TFBertModel

print(f"TensorFlow {tf.__version__}")

# Rutas
RUTA_CSV = "/home/esebas/Escritorio/Sebas/IA SAMSUNG/PROYECTO/datasets/text_and_description_dataset.csv"
RUTA_IMAGENES = "/home/esebas/Escritorio/Sebas/IA SAMSUNG/PROYECTO/extraccion memes/ALL MEMES"

# Hiperparámetros principales
MAX_LEN = 128
BATCH_SIZE = 16
IMG_SIZE = 224
EPOCHS = 50

INIT_LR = 1.5e-5
MIN_LR = 1e-7

# Regularización
L2_REG = 5e-5
DROPOUT_VISUAL = 0.55
DROPOUT_TEXTUAL = 0.55
DROPOUT_CLASSIFIER = 0.45
DROPOUT_FINAL = 0.35

# Fine-tuning
CNN_TRAINABLE_LAYERS = 15
BERT_TRAINABLE_LAYERS = 1

# Data augmentation
AUG_ROTATION = 0.2
AUG_ZOOM = 0.2
AUG_BRIGHTNESS = 0.25
AUG_CONTRAST = 0.15

2026-02-02 17:17:56.017041: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow 2.20.0


In [2]:
# Carga del dataset
df = pd.read_csv(RUTA_CSV)

required_columns = ['meme_id', 'extracted_text', 'description_es_new', 'harmless']
for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Columna faltante: {col}")

df['extracted_text'] = df['extracted_text'].fillna('')
df['description_es_new'] = df['description_es_new'].fillna('')

# Texto combinado imagen + descripción
df['text_final'] = df.apply(
    lambda r: f"MEME: {r['extracted_text'][:150]} [SEP] ESCENA: {r['description_es_new'][:200]}",
    axis=1
)

In [3]:
# Split estratificado
df_train, df_val = train_test_split(
    df,
    test_size=0.25,
    stratify=df['harmless'],
    random_state=42
)

In [4]:
# Tokenización BERT
tokenizer = AutoTokenizer.from_pretrained(
    'dccuchile/bert-base-spanish-wwm-cased'
)

def tokenize(texts, batch_size=64):
    ids, masks = [], []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i+batch_size],
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="tf"
        )
        ids.append(enc['input_ids'])
        masks.append(enc['attention_mask'])
    return {
        'input_ids': tf.concat(ids, axis=0),
        'attention_mask': tf.concat(masks, axis=0)
    }

t_train = tokenize(df_train['text_final'].tolist())
t_val = tokenize(df_val['text_final'].tolist())

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
2026-02-02 17:18:43.635550: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1770049123.636491   45407 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [5]:
# Data augmentation para entrenamiento
data_aug = tf.keras.Sequential([
    layers.RandomRotation(AUG_ROTATION),
    layers.RandomZoom(AUG_ZOOM),
    layers.RandomBrightness(AUG_BRIGHTNESS),
    layers.RandomContrast(AUG_CONTRAST),
    layers.RandomFlip("horizontal"),
    layers.GaussianNoise(0.01)
])

In [6]:
# Pipeline de imágenes
def build_pipeline(augment=False):
    def process(path, ids, mask, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        if augment:
            img = data_aug(tf.expand_dims(img, 0))[0]
        img = tf.keras.applications.efficientnet.preprocess_input(img)
        return {"img_input": img, "ids_input": ids, "mask_input": mask}, label
    return process

AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((
    [os.path.join(RUTA_IMAGENES, str(m)) for m in df_train['meme_id']],
    t_train['input_ids'],
    t_train['attention_mask'],
    df_train['harmless'].astype('float32')
)).shuffle(len(df_train)).map(
    build_pipeline(augment=True),
    num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((
    [os.path.join(RUTA_IMAGENES, str(m)) for m in df_val['meme_id']],
    t_val['input_ids'],
    t_val['attention_mask'],
    df_val['harmless'].astype('float32')
)).map(
    build_pipeline(augment=False),
    num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [7]:
# Arquitectura balanceada
def build_balanced_model():
    img_in = layers.Input((IMG_SIZE, IMG_SIZE, 3), name='img_input')
    ids_in = layers.Input((MAX_LEN,), dtype=tf.int32, name='ids_input')
    mask_in = layers.Input((MAX_LEN,), dtype=tf.int32, name='mask_input')

    cnn = EfficientNetB0(weights='imagenet', include_top=False)
    for l in cnn.layers[:-CNN_TRAINABLE_LAYERS]:
        l.trainable = False

    x_img = cnn(img_in)
    x_img = layers.Concatenate()([
        layers.GlobalAveragePooling2D()(x_img),
        layers.GlobalMaxPooling2D()(x_img)
    ])
    x_img = layers.Dense(
        192, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG)
    )(x_img)
    x_img = layers.BatchNormalization()(x_img)
    x_img = layers.Dropout(DROPOUT_VISUAL)(x_img)

    bert = TFBertModel.from_pretrained(
        'dccuchile/bert-base-spanish-wwm-cased'
    )
    for l in bert.bert.encoder.layer[:-BERT_TRAINABLE_LAYERS]:
        l.trainable = False

    b = bert(ids_in, attention_mask=mask_in)
    x_txt = layers.Concatenate()([b.pooler_output, b.last_hidden_state[:, 0]])
    x_txt = layers.Dense(
        192, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG)
    )(x_txt)
    x_txt = layers.BatchNormalization()(x_txt)
    x_txt = layers.Dropout(DROPOUT_TEXTUAL)(x_txt)

    inter = layers.Multiply()([x_img, x_txt])
    diff = layers.Subtract()([x_img, x_txt])

    fused = layers.Concatenate()([x_img, x_txt, inter, diff])

    x = layers.Dense(
        128, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG * 0.5)
    )(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_CLASSIFIER)(x)

    x = layers.Dense(
        64, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG * 0.2)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_FINAL)(x)

    out = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(
        inputs=[img_in, ids_in, mask_in],
        outputs=out,
        name='modelo_MEMES4GOOD_v4'
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=INIT_LR,
            clipnorm=1.0
        ),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )
    return model

model = build_balanced_model()
model.summary()

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing TFBertModel: ['mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFBertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert/pooler/dense/kernel:0', 'bert/pooler/dense/bias:0']
You should probably TRAIN this model on

Model: "modelo_MEMES4GOOD_v4"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 img_input (InputLayer)      [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 ids_input (InputLayer)      [(None, 128)]                0         []                            
                                                                                                  
 mask_input (InputLayer)     [(None, 128)]                0         []                            
                                                                                                  
 efficientnetb0 (Functional  (None, None, None, 1280)     4049571   ['img_input[0][0]']           
 )                                                                             

In [8]:
# Pesos de clase
weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(df_train['harmless']),
    y=df_train['harmless']
)
class_weights = dict(enumerate(weights))

In [10]:
# Callbacks
callbacks_list = [
    callbacks.ModelCheckpoint(
        'modelo_memes4good_BALANCED.keras',
        monitor='val_auc',
        save_best_only=True,
        mode='max'
    ),
    callbacks.EarlyStopping(
        monitor='val_auc',
        patience=15,
        restore_best_weights=True,
        mode='max'
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=MIN_LR
    )
]

In [11]:
# Entrenamiento
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_list
)

#Guardo modelo final
model.save('modelo_memes4good_BALANCED.keras')
pd.DataFrame(history.history).to_csv(
    'training_history_balanced.csv',
    index=False
)

Epoch 1/50


E0000 00:00:1770049574.871753   45407 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inmodelo_MEMES4GOOD_v4/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-02-02 17:26:17.080759: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-02-02 17:26:19.144650: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d0a74483800 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-02 17:26:19.144669: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2026-02-02 17:26:19.159870: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1770049579.239075   45508 device_compiler.h:196] Compiled cl

319/319 [==============================] - 110s 287ms/step - loss: 1.0358 - accuracy: 0.5033 - auc: 0.5147 - precision: 0.3887 - recall: 0.5233 - val_loss: 0.8746 - val_accuracy: 0.4553 - val_auc: 0.5655 - val_precision: 0.3934 - val_recall: 0.7874 - lr: 1.5000e-05
Epoch 2/50
319/319 [==============================] - 88s 275ms/step - loss: 1.0106 - accuracy: 0.5227 - auc: 0.5291 - precision: 0.4049 - recall: 0.5300 - val_loss: 0.8764 - val_accuracy: 0.4971 - val_auc: 0.5979 - val_precision: 0.4161 - val_recall: 0.7874 - lr: 1.5000e-05
Epoch 3/50
319/319 [==============================] - 88s 277ms/step - loss: 0.9861 - accuracy: 0.5284 - auc: 0.5439 - precision: 0.4109 - recall: 0.5393 - val_loss: 0.8985 - val_accuracy: 0.5024 - val_auc: 0.6414 - val_precision: 0.4241 - val_recall: 0.8475 - lr: 1.5000e-05
Epoch 4/50
319/319 [==============================] - 89s 278ms/step - loss: 0.9158 - accuracy: 0.5553 - auc: 0.5880 - precision: 0.4380 - recall: 0.5782 - val_loss: 0.7895 - val_acc